# DTLS — CoAP lewat coaps://

Node rumah kaca (`CoapServer`) dan klien berbicara CoAP lewat DTLS 1.2 di jaringan datagram dalam memori: pertama dengan pre-shared key, lalu dengan sertifikat ber-pin. Untuk perangkat sungguhan, ganti `net.Bind(...)` dengan `new UdpDatagramTransport(...)` dan pakai port 5684.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.29.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.29.0-preview.1"
#r "nuget: IoTCom.Net.Transport.Dtls, 0.29.0-preview.1"

## Pre-shared key
Profil RFC 7925 untuk perangkat terbatas: TLS_PSK_WITH_AES_128_CCM_8. Server mencari kunci berdasarkan identitas; kunci tidak pernah muncul di log atau `ToString()`.

In [ ]:
using System.Net;
using IoTCom.Net;
using IoTCom.Net.Protocols.Coap;
using IoTCom.Net.Transport.Dtls;
using IoTCom.Net.Transports;

var net = new InMemoryDatagramNetwork();
var at = new IPEndPoint(IPAddress.Parse("10.0.0.50"), 5684);
var psk = DtlsPsk.FromHex("greenhouse-node-01", "6A2F0C9B1D4E7F3A8B5C2D1E0F9A8B7C");
var dtlsServer = new DtlsServerTransport(net.Bind(at), new DtlsServerOptions().UsePsk(psk));
dtlsServer.SessionEstablished += s => Console.WriteLine($"server: session with {s.Remote}, {s.CipherSuite}, identity {s.PskIdentity}");
var node = CoapServer.Create(o => o.UseDatagramTransport(() => dtlsServer));
var device = new CoapDeviceSimulator(node);
await node.StartAsync();

var client = CoapClient.Create(o => o.UseDtlsClient(() => net.Bind(), at, new DtlsClientOptions { Psk = psk }).UseServer(at));
await client.ConnectAsync();
Console.WriteLine($"temperature {(await client.GetAsync("/sensors/temperature")).PayloadText} °C  (key: {psk})");

## Kunci yang salah
Record dengan MAC salah dibuang diam-diam (RFC 6347), jadi kunci yang salah berakhir dengan timeout handshake beserta petunjuk.

In [ ]:
var wrong = new DtlsClientTransport(net.Bind(), at, new DtlsClientOptions { Psk = DtlsPsk.FromHex("greenhouse-node-01", "00112233445566778899AABBCCDDEEFF"), HandshakeTimeout = TimeSpan.FromSeconds(2) });
try { await wrong.HandshakeAsync(); } catch (IoTComException e) { Console.WriteLine(e.Message); }
await wrong.DisposeAsync();
await client.DisposeAsync();
await device.DisposeAsync();
await node.DisposeAsync();

## Sertifikat dan pinning
ECDHE_ECDSA dengan sertifikat P-256 self-signed; klien hanya menerima sidik jari yang di-pin.

In [ ]:
var cert = DtlsCertificates.CreateSelfSigned("greenhouse-gw.local");
var gw = CoapServer.Create(o => o.UseDtlsServer(() => net.Bind(new IPEndPoint(IPAddress.Parse("10.0.0.51"), 5684)), new DtlsServerOptions { Certificate = cert }));
var gwDevice = new CoapDeviceSimulator(gw);
await gw.StartAsync();
var gwAt = new IPEndPoint(IPAddress.Parse("10.0.0.51"), 5684);
var pinned = new DtlsClientOptions();
pinned.PinnedFingerprints.Add(DtlsCertificates.Fingerprint(cert));
var secure = new DtlsClientTransport(net.Bind(), gwAt, pinned);
await secure.HandshakeAsync();
Console.WriteLine($"suite {secure.CipherSuite}, pin {DtlsCertificates.Fingerprint(cert)[..16]}…");
var gwClient = CoapClient.Create(o => o.UseDatagramTransport(() => secure).UseServer(gwAt));
await gwClient.ConnectAsync();
Console.WriteLine($"soil {(await gwClient.GetAsync("/sensors/soil")).PayloadText} %");

var other = new DtlsClientOptions();
other.PinnedFingerprints.Add(new string('0', 64));
var intruder = new DtlsClientTransport(net.Bind(), gwAt, other);
try { await intruder.HandshakeAsync(); } catch (IoTComException e) { Console.WriteLine(e.Message); }
await intruder.DisposeAsync();
await gwClient.DisposeAsync();
await gwDevice.DisposeAsync();
await gw.DisposeAsync();

## Lebih lanjut
`iotcom coap serve --psk id:hex` atau `--cert`, `iotcom coap get coaps://host/path --psk id:hex` (atau `IOTCOM_DTLS_PSK`), `--pin <sha256>`, dan sampel CoapsSecure. LwM2M memakai `UseDtlsClient`/`UseDtlsServer` yang sama. Lihat `docs/id/protocols/dtls.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*